# Неравновесное изотермическое сжатие LJ-флюида

Этот Kaggle-notebook проводит один непрерывный MD-прогон изотермического сжатия Lennard-Jones-флюида при `T = 0.8`, сохраняет численные данные и координаты кадров, а затем отдельно рендерит MP4.

Ролик показывает **неравновесное изотермическое сжатие при конечной скорости**, а не последовательность равновесных состояний. Стационарная MD-изотерма `P(rho)` используется только как фон и ориентир; динамическая траектория может отклоняться от неё из-за запаздывания релаксации, метастабильности и гистерезиса.

Notebook не строит fit Ван-дер-Ваальса, бинодаль или спинодаль, не сохраняет DCD и не использует multi-GPU.


In [ ]:
# Установка и импорт зависимостей для Kaggle GPU.

import os, sys, math, time, json, csv, subprocess, shutil
from pathlib import Path
from statistics import mean, stdev

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openmm[cuda12]', 'numpy', 'pandas', 'matplotlib', 'pillow', 'pyvista', 'imageio', 'tqdm'
])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

import openmm as mm
import openmm.app as app
from openmm import unit

import pyvista as pv
pv.OFF_SCREEN = True

print('Python:', sys.version.split()[0])
print('OpenMM:', mm.version.version)
print('PyVista:', pv.__version__)
print('\n===== nvidia-smi =====')
try:
    smi = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
    print(smi.stdout if smi.stdout else smi.stderr)
except FileNotFoundError:
    print('nvidia-smi not found')

OPENMM_PLATFORMS = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print('OpenMM platforms:', OPENMM_PLATFORMS)
if 'CUDA' not in OPENMM_PLATFORMS:
    raise RuntimeError('CUDA is not available in OpenMM. This notebook is intended for Kaggle GPU runs.')

ffmpeg_check = subprocess.run(['ffmpeg', '-version'], text=True, capture_output=True)
print('ffmpeg available:', ffmpeg_check.returncode == 0)


In [ ]:
# Конфигурация симуляции, вывода и рендера.

RUN_SIMULATION = True
RUN_RENDERING = True
KEEP_RENDERED_FRAMES = False
SMOKE_TEST = False
MAX_RENDER_FRAMES = 30 if SMOKE_TEST else None

OUTPUT_DIR = Path('compression_movie_data_smoke' if SMOKE_TEST else 'compression_movie_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR = OUTPUT_DIR / 'rendered_frames'

TRACE_CSV = OUTPUT_DIR / 'compression_trace.csv'
FRAMES_NPZ = OUTPUT_DIR / 'compression_frames.npz'
METADATA_JSON = OUTPUT_DIR / 'compression_metadata.json'
MP4_PATH = OUTPUT_DIR / 'lj_compression_T080.mp4'

# Укажите путь к EOS CSV. Для Kaggle удобнее загрузить отдельный файл
# eos_for_compression_movie_T080.csv как input dataset; он стоит первым в поиске.
EOS_FILES_GLOB = [
    'local_data/processed/eos_for_compression_movie_T080.csv',
    '/kaggle/input/**/eos_for_compression_movie_T080.csv',
    'local_data/raw/eos_seed_*.csv',
    'local_data/raw/eos_targeted*.csv',
    'data/eos_seed_*.csv',
    'data/eos_targeted*.csv',
    '/kaggle/input/**/*.csv',
]
EOS_T_TARGET = 0.8
EOS_T_TOL = 1e-8

N = 512
T_TARGET = 0.8
RHO_START = 0.05
RHO_END = 0.85

SIGMA = 1.0
EPSILON = 1.0
MASS = 1.0
RCUT = 2.5
DT = 0.002
FRICTION = 1.0
SEED = 20260620

INITIAL_EQUIL_STEPS = 20_000
FINAL_HOLD_STEPS = 10_000
COMPRESSION_SEGMENTS = [
    {'name': 'Разреженный газ', 'rho_from': 0.05, 'rho_to': 0.15, 'n_stages': 20, 'steps_per_stage': 500},
    {'name': 'Кластеризация и неустойчивая область', 'rho_from': 0.15, 'rho_to': 0.55, 'n_stages': 60, 'steps_per_stage': 1_500},
    {'name': 'Переход к плотной жидкости', 'rho_from': 0.55, 'rho_to': 0.85, 'n_stages': 30, 'steps_per_stage': 750},
]
FINAL_HOLD_SEGMENT_NAME = 'Выдержка при конечной плотности'

MEASUREMENT_INTERVAL = 100
FRAME_INTERVAL = 250
PRESSURE_SMOOTHING_WINDOW_STEPS = 2_500

DEVICE_INDEX = '0'
PRECISION = 'mixed'

VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720
LEFT_WIDTH = 860
RIGHT_WIDTH = VIDEO_WIDTH - LEFT_WIDTH
FPS = 20
HOLD_SECONDS = 1
POINT_SIZE = 14
BACKGROUND_COLOR = '#f7f7f4'

if SMOKE_TEST:
    N = 64
    RHO_START = 0.10
    RHO_END = 0.30
    INITIAL_EQUIL_STEPS = 100
    FINAL_HOLD_STEPS = 100
    COMPRESSION_SEGMENTS = [
        {'name': 'Smoke compression', 'rho_from': 0.10, 'rho_to': 0.30, 'n_stages': 3, 'steps_per_stage': 100},
    ]
    MEASUREMENT_INTERVAL = 10
    FRAME_INTERVAL = 20
    PRESSURE_SMOOTHING_WINDOW_STEPS = 50

print('OUTPUT_DIR:', OUTPUT_DIR)
print('RUN_SIMULATION:', RUN_SIMULATION)
print('RUN_RENDERING:', RUN_RENDERING)
print('SMOKE_TEST:', SMOKE_TEST, 'MAX_RENDER_FRAMES:', MAX_RENDER_FRAMES)
print('N:', N, 'T:', T_TARGET, 'rho:', RHO_START, '->', RHO_END)
print('Expected stages:', sum(int(s['n_stages']) for s in COMPRESSION_SEGMENTS))


In [ ]:
# Проверенные LJ/OpenMM-функции, совместимые с остальными notebook проекта.

EOS_FIELDS_REQUIRED = {'T_target', 'rho_target', 'P_mean'}


def run_id_for_frame(step):
    return 'compression_step_' + str(int(step))


def openmm_temperature(T):
    gas_r = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole / unit.kelvin)
    return (float(T) * EPSILON / gas_r) * unit.kelvin


def initialize_positions(N, L):
    n_side = math.ceil(int(N) ** (1.0 / 3.0))
    spacing = float(L) / n_side
    coords = []
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                if len(coords) == int(N):
                    return np.asarray(coords, dtype=float)
                coords.append([(ix + 0.5) * spacing, (iy + 0.5) * spacing, (iz + 0.5) * spacing])
    return np.asarray(coords, dtype=float)


def make_topology(N):
    topology = app.Topology()
    chain = topology.addChain()
    residue = topology.addResidue('LJ', chain)
    element = app.Element.getByAtomicNumber(18)
    for _ in range(int(N)):
        topology.addAtom('Ar', element, residue)
    return topology


def create_lj_system(N, rho):
    L = float((int(N) / float(rho)) ** (1.0 / 3.0))
    system = mm.System()
    for _ in range(int(N)):
        system.addParticle(MASS * unit.dalton)
    system.setDefaultPeriodicBoxVectors(
        unit.Quantity(mm.Vec3(L, 0, 0), unit.nanometer),
        unit.Quantity(mm.Vec3(0, L, 0), unit.nanometer),
        unit.Quantity(mm.Vec3(0, 0, L), unit.nanometer),
    )
    expr = '4*epsilon*((sigma/r)^12-(sigma/r)^6)-4*epsilon*((sigma/rcut)^12-(sigma/rcut)^6)'
    force = mm.CustomNonbondedForce(expr)
    force.addGlobalParameter('sigma', SIGMA * unit.nanometer)
    force.addGlobalParameter('epsilon', EPSILON * unit.kilojoule_per_mole)
    force.addGlobalParameter('rcut', RCUT * unit.nanometer)
    force.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    force.setCutoffDistance(RCUT * unit.nanometer)
    force.setUseLongRangeCorrection(False)
    for _ in range(int(N)):
        force.addParticle([])
    system.addForce(force)
    return system, L


def create_cuda_platform():
    platform = mm.Platform.getPlatformByName('CUDA')
    properties = {'DeviceIndex': str(DEVICE_INDEX), 'Precision': str(PRECISION)}
    return platform, properties


def get_positions(simulation):
    state = simulation.context.getState(getPositions=True)
    return np.asarray(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer), dtype=float)


def lj_virial_pressure(positions, L, T):
    V = float(L) ** 3
    rho = len(positions) / V
    virial = 0.0
    rcut2 = RCUT * RCUT
    for i in range(len(positions) - 1):
        delta = positions[i + 1:] - positions[i]
        delta -= L * np.rint(delta / L)
        r2 = np.sum(delta * delta, axis=1)
        mask = (r2 > 0.0) & (r2 < rcut2)
        if not np.any(mask):
            continue
        inv_r2 = (SIGMA * SIGMA) / r2[mask]
        inv_r6 = inv_r2 ** 3
        inv_r12 = inv_r6 ** 2
        virial += float(np.sum(24.0 * EPSILON * (2.0 * inv_r12 - inv_r6)))
    return float(rho * T + virial / (3.0 * V))


def sample_observables(simulation, step, segment_name, stage_index, rho, L):
    state = simulation.context.getState(getEnergy=True)
    K_total = state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
    U_total = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    T_inst = 2.0 * K_total / (max(1, 3 * N - 3) * EPSILON)
    positions = get_positions(simulation)
    P_inst = lj_virial_pressure(positions, L, T_inst)
    U_per_particle = float(U_total) / int(N)
    K_per_particle = float(K_total) / int(N)
    if not all(math.isfinite(x) for x in [T_inst, P_inst, U_per_particle, K_per_particle]):
        raise RuntimeError('Non-finite observable at step ' + str(step))
    if T_inst > 10.0 * float(T_TARGET):
        raise RuntimeError('Unstable temperature at step ' + str(step) + ': ' + str(T_inst))
    return {
        'step': int(step),
        'time': float(step) * DT,
        'segment': str(segment_name),
        'stage': int(stage_index),
        'rho': float(rho),
        'box_length': float(L),
        'T_instant': float(T_inst),
        'P_instant': float(P_inst),
        'U_per_particle': float(U_per_particle),
        'K_per_particle': float(K_per_particle),
        'E_per_particle': float(U_per_particle + K_per_particle),
    }


def set_box_and_scaled_positions(simulation, old_L, new_L):
    positions = get_positions(simulation)
    scaled = positions * (float(new_L) / float(old_L))
    scaled = np.mod(scaled, float(new_L))
    simulation.context.setPeriodicBoxVectors(
        mm.Vec3(new_L, 0, 0) * unit.nanometer,
        mm.Vec3(0, new_L, 0) * unit.nanometer,
        mm.Vec3(0, 0, new_L) * unit.nanometer,
    )
    simulation.context.setPositions(scaled * unit.nanometer)
    return scaled


def cuda_context_check():
    old_N = globals()['N']
    try:
        globals()['N'] = 8
        system, L = create_lj_system(N, 0.05)
        target_T = openmm_temperature(1.0)
        integrator = mm.LangevinMiddleIntegrator(target_T, FRICTION / unit.picosecond, DT * unit.picoseconds)
        integrator.setRandomNumberSeed(123)
        platform, properties = create_cuda_platform()
        sim = app.Simulation(make_topology(N), system, integrator, platform, properties)
        sim.context.setPositions(initialize_positions(N, L) * unit.nanometer)
        sim.context.setVelocitiesToTemperature(target_T, 123)
        sim.step(2)
        actual = sim.context.getPlatform().getName()
        if actual != 'CUDA':
            raise RuntimeError('Expected CUDA platform, got ' + actual)
        print('CUDA Context check ok:', actual, properties)
    finally:
        globals()['N'] = old_N

cuda_context_check()


In [ ]:
# Загрузка стационарной MD-изотермы T = 0.8 для правой панели ролика.


def expand_globs(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]
    paths = []
    for pattern in patterns:
        paths.extend(Path('.').glob(pattern) if not str(pattern).startswith('/') else Path('/').glob(str(pattern).lstrip('/')))
    return sorted(dict.fromkeys([p for p in paths if p.is_file()]))


def load_stationary_isotherm():
    paths = expand_globs(EOS_FILES_GLOB)
    print('EOS files found:', [str(p) for p in paths[:30]], '...' if len(paths) > 30 else '')
    frames = []
    for path in paths:
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if not EOS_FIELDS_REQUIRED.issubset(df.columns):
            continue
        df = df.copy()
        df['source_file'] = str(path)
        frames.append(df)
    if not frames:
        raise RuntimeError('No usable EOS CSV found. Set EOS_FILES_GLOB to files containing T_target, rho_target and P_mean.')
    eos = pd.concat(frames, ignore_index=True)
    for col in ['seed', 'T_target', 'rho_target', 'P_mean', 'P_std']:
        if col in eos.columns:
            eos[col] = pd.to_numeric(eos[col], errors='coerce')
    if 'status' in eos.columns:
        eos = eos[eos['status'].fillna('ok') == 'ok']
    eos_T = eos[np.isclose(eos['T_target'], EOS_T_TARGET, atol=EOS_T_TOL)].copy()
    if len(eos_T) == 0:
        raise RuntimeError('EOS files were found, but no rows with T_target = ' + str(EOS_T_TARGET) + '. Check EOS_FILES_GLOB or EOS_T_TARGET.')
    if 'seed' in eos_T.columns:
        eos_points = eos_T.groupby(['seed', 'rho_target'], as_index=False).agg(P_mean=('P_mean', 'mean'))
        iso = eos_points.groupby('rho_target').agg(P_mean=('P_mean', 'mean'), P_std=('P_mean', 'std'), n_seeds=('seed', 'nunique')).reset_index()
    else:
        iso = eos_T.groupby('rho_target').agg(P_mean=('P_mean', 'mean'), P_std=('P_mean', 'std'), n_seeds=('P_mean', 'size')).reset_index()
    iso['P_std'] = iso['P_std'].fillna(0.0)
    iso = iso.sort_values('rho_target')
    if len(iso) < 3:
        raise RuntimeError('Too few EOS points at T = ' + str(EOS_T_TARGET) + ' for the stationary isotherm panel.')
    return iso

stationary_isotherm = load_stationary_isotherm()
print('Stationary isotherm rows:', len(stationary_isotherm))
display(stationary_isotherm.head())


In [ ]:
# MD-симуляция ступенчатого изотермического сжатия и сохранение CSV/NPZ.


def build_schedule():
    schedule = []
    stage_index = 0
    for segment in COMPRESSION_SEGMENTS:
        rhos = np.linspace(float(segment['rho_from']), float(segment['rho_to']), int(segment['n_stages']) + 1)[1:]
        for rho in rhos:
            stage_index += 1
            schedule.append({
                'segment': str(segment['name']),
                'stage': int(stage_index),
                'rho': float(rho),
                'steps': int(segment['steps_per_stage']),
            })
    if FINAL_HOLD_STEPS:
        stage_index += 1
        schedule.append({'segment': FINAL_HOLD_SEGMENT_NAME, 'stage': int(stage_index), 'rho': float(RHO_END), 'steps': int(FINAL_HOLD_STEPS)})
    return schedule


def save_metadata(actual_platform, simulation_wall_time_s, total_steps, n_frames):
    metadata = {
        'N': int(N),
        'T': float(T_TARGET),
        'rho_start': float(RHO_START),
        'rho_end': float(RHO_END),
        'dt': float(DT),
        'rcut': float(RCUT),
        'initial_equil_steps': int(INITIAL_EQUIL_STEPS),
        'compression_segments': COMPRESSION_SEGMENTS,
        'final_hold_steps': int(FINAL_HOLD_STEPS),
        'measurement_interval': int(MEASUREMENT_INTERVAL),
        'frame_interval': int(FRAME_INTERVAL),
        'OpenMM_version': str(mm.version.version),
        'platform': str(actual_platform),
        'device_index': str(DEVICE_INDEX),
        'precision': str(PRECISION),
        'seed': int(SEED),
        'simulation_wall_time_s': float(simulation_wall_time_s),
        'total_md_steps': int(total_steps),
        'n_frames': int(n_frames),
    }
    METADATA_JSON.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')


def maybe_record_frame(simulation, frames, frame_steps, frame_rhos, frame_Ls, frame_segments, step, rho, L, segment):
    positions = get_positions(simulation).astype(np.float32)
    if np.nanmin(positions) < -1e-5 or np.nanmax(positions) > float(L) + 1e-5:
        raise RuntimeError('Positions outside periodic cell at step ' + str(step))
    frames.append(positions)
    frame_steps.append(int(step))
    frame_rhos.append(float(rho))
    frame_Ls.append(float(L))
    frame_segments.append(str(segment))


def run_compression_simulation():
    system, L = create_lj_system(N, RHO_START)
    target_T = openmm_temperature(T_TARGET)
    integrator = mm.LangevinMiddleIntegrator(target_T, FRICTION / unit.picosecond, DT * unit.picoseconds)
    integrator.setRandomNumberSeed(int(SEED))
    platform, properties = create_cuda_platform()
    simulation = app.Simulation(make_topology(N), system, integrator, platform, properties)
    simulation.context.setPositions(initialize_positions(N, L) * unit.nanometer)
    simulation.context.setVelocitiesToTemperature(target_T, int(SEED))
    actual_platform = simulation.context.getPlatform().getName()
    if actual_platform != 'CUDA':
        raise RuntimeError('Expected CUDA platform, got ' + actual_platform)

    trace_rows = []
    frames = []
    frame_steps = []
    frame_rhos = []
    frame_Ls = []
    frame_segments = []

    start = time.perf_counter()
    current_step = 0
    current_rho = float(RHO_START)

    print('Initial equilibration steps:', INITIAL_EQUIL_STEPS)
    while current_step < INITIAL_EQUIL_STEPS:
        chunk = min(MEASUREMENT_INTERVAL, INITIAL_EQUIL_STEPS - current_step)
        simulation.step(int(chunk))
        current_step += int(chunk)
    trace_rows.append(sample_observables(simulation, current_step, 'Начальное равновесие', 0, current_rho, L))
    maybe_record_frame(simulation, frames, frame_steps, frame_rhos, frame_Ls, frame_segments, current_step, current_rho, L, 'Начальное равновесие')

    schedule = build_schedule()
    print('Compression/hold stages:', len(schedule))
    for item in tqdm(schedule, desc='compression stages'):
        old_L = float(L)
        current_rho = float(item['rho'])
        L = float((int(N) / current_rho) ** (1.0 / 3.0))
        if abs(L - old_L) > 1e-12:
            set_box_and_scaled_positions(simulation, old_L, L)
        stage_remaining = int(item['steps'])
        stage_done = 0
        while stage_done < stage_remaining:
            next_measure = MEASUREMENT_INTERVAL - (current_step % MEASUREMENT_INTERVAL)
            next_frame = FRAME_INTERVAL - (current_step % FRAME_INTERVAL)
            chunk = min(stage_remaining - stage_done, next_measure, next_frame)
            simulation.step(int(chunk))
            current_step += int(chunk)
            stage_done += int(chunk)
            if current_step % MEASUREMENT_INTERVAL == 0 or stage_done == stage_remaining:
                trace_rows.append(sample_observables(simulation, current_step, item['segment'], item['stage'], current_rho, L))
            if current_step % FRAME_INTERVAL == 0 or stage_done == stage_remaining:
                maybe_record_frame(simulation, frames, frame_steps, frame_rhos, frame_Ls, frame_segments, current_step, current_rho, L, item['segment'])

    simulation_wall_time_s = time.perf_counter() - start
    trace = pd.DataFrame(trace_rows)
    trace.to_csv(TRACE_CSV, index=False)
    np.savez_compressed(
        FRAMES_NPZ,
        positions=np.asarray(frames, dtype=np.float32),
        step=np.asarray(frame_steps, dtype=np.int64),
        rho=np.asarray(frame_rhos, dtype=np.float32),
        box_length=np.asarray(frame_Ls, dtype=np.float32),
        segment=np.asarray(frame_segments, dtype=object),
    )
    save_metadata(actual_platform, simulation_wall_time_s, current_step, len(frames))
    print('saved:', TRACE_CSV)
    print('saved:', FRAMES_NPZ)
    print('saved:', METADATA_JSON)
    return trace

if RUN_SIMULATION:
    trace_df = run_compression_simulation()
else:
    if not TRACE_CSV.exists() or not FRAMES_NPZ.exists():
        raise RuntimeError('RUN_SIMULATION=False, but saved trace/frames were not found in ' + str(OUTPUT_DIR))
    trace_df = pd.read_csv(TRACE_CSV)
    print('Loaded existing simulation data:', TRACE_CSV, FRAMES_NPZ)


In [ ]:
# Загрузка сохранённых данных, сглаживание давления и служебные графики.

trace_df = pd.read_csv(TRACE_CSV)
frames_data = np.load(FRAMES_NPZ, allow_pickle=True)
frame_positions = frames_data['positions']
frame_steps = frames_data['step']
frame_rhos = frames_data['rho']
frame_Ls = frames_data['box_length']
frame_segments = frames_data['segment']

window_n = max(1, int(round(PRESSURE_SMOOTHING_WINDOW_STEPS / MEASUREMENT_INTERVAL)))
if window_n % 2 == 0:
    window_n += 1
trace_df['P_smooth'] = trace_df['P_instant'].rolling(window=window_n, center=True, min_periods=1).mean()
trace_df.to_csv(TRACE_CSV, index=False)

print('density min/max:', float(trace_df['rho'].min()), float(trace_df['rho'].max()))
print('temperature min/max/mean:', float(trace_df['T_instant'].min()), float(trace_df['T_instant'].max()), float(trace_df['T_instant'].mean()))
print('smoothed pressure min/max:', float(trace_df['P_smooth'].min()), float(trace_df['P_smooth'].max()))
print('frames:', len(frame_steps))
print('total MD steps:', int(trace_df['step'].max()))

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes[0, 0].plot(trace_df['step'], trace_df['rho'])
axes[0, 0].set_xlabel('step'); axes[0, 0].set_ylabel('rho')
axes[0, 1].plot(trace_df['step'], trace_df['T_instant'])
axes[0, 1].axhline(T_TARGET, color='black', linewidth=1)
axes[0, 1].set_xlabel('step'); axes[0, 1].set_ylabel('T')
axes[1, 0].plot(trace_df['step'], trace_df['P_instant'], alpha=0.35, label='instant')
axes[1, 0].plot(trace_df['step'], trace_df['P_smooth'], label='smooth')
axes[1, 0].set_xlabel('step'); axes[1, 0].set_ylabel('P'); axes[1, 0].legend()
axes[1, 1].plot(trace_df['step'], trace_df['box_length'])
axes[1, 1].set_xlabel('step'); axes[1, 1].set_ylabel('box length')
for ax in axes.ravel():
    ax.grid(alpha=0.3)
fig.tight_layout()
path = OUTPUT_DIR / 'compression_diagnostics.png'
fig.savefig(path, dpi=150)
plt.close(fig)
print('saved:', path)


In [ ]:
# Функции рендеринга 3D-панели, EOS-панели и итоговых кадров.


def pressure_at_step(step):
    idx = int(np.argmin(np.abs(trace_df['step'].to_numpy() - int(step))))
    row = trace_df.iloc[idx]
    return float(row['P_smooth']), float(row['T_instant']), str(row['segment'])


def trajectory_until_step(step):
    part = trace_df[trace_df['step'] <= int(step)]
    return part['rho'].to_numpy(), part['P_smooth'].to_numpy()


def render_box_edges(plotter, L):
    half = float(L) / 2.0
    corners = np.array([
        [-half, -half, -half], [ half, -half, -half], [ half,  half, -half], [-half,  half, -half],
        [-half, -half,  half], [ half, -half,  half], [ half,  half,  half], [-half,  half,  half],
    ], dtype=float)
    edges = [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4), (0,4), (1,5), (2,6), (3,7)]
    for a, b in edges:
        line = pv.Line(corners[a], corners[b])
        plotter.add_mesh(line, color='#555555', line_width=1.4)


def render_3d_panel(positions, L):
    plotter = pv.Plotter(off_screen=True, window_size=(LEFT_WIDTH, VIDEO_HEIGHT))
    plotter.set_background(BACKGROUND_COLOR)
    view_positions = np.asarray(positions, dtype=float) - float(L) / 2.0
    cloud = pv.PolyData(view_positions)
    plotter.add_mesh(
        cloud,
        color='#2a6fbb',
        point_size=POINT_SIZE,
        render_points_as_spheres=True,
        ambient=0.35,
        diffuse=0.75,
    )
    render_box_edges(plotter, L)
    initial_L = float((int(N) / float(RHO_START)) ** (1.0 / 3.0))
    distance = initial_L * 1.65
    plotter.camera_position = [
        (distance, -distance, distance * 0.55),
        (0.0, 0.0, 0.0),
        (0.0, 0.0, 1.0),
    ]
    bound = initial_L * 0.72
    plotter.show_bounds(
        bounds=(-bound, bound, -bound, bound, -bound, bound),
        grid=False, location='outer', all_edges=False, xtitle='', ytitle='', ztitle='',
        font_size=1, color=BACKGROUND_COLOR,
    )
    plotter.remove_bounds_axes()
    img = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(img).resize((LEFT_WIDTH, VIDEO_HEIGHT))


def render_plot_panel(step, rho, P_smooth, T_inst, segment):
    fig, ax = plt.subplots(figsize=(RIGHT_WIDTH / 100, VIDEO_HEIGHT / 100), dpi=100)
    fig.patch.set_facecolor('white')
    ax.errorbar(
        stationary_isotherm['rho_target'], stationary_isotherm['P_mean'],
        yerr=stationary_isotherm['P_std'], color='#777777', marker='o', markersize=3,
        linewidth=1.2, capsize=2, label='Стационарная MD-изотерма'
    )
    ax.axhline(0.0, color='black', linewidth=0.8, alpha=0.7)
    tr_rho, tr_P = trajectory_until_step(step)
    ax.plot(tr_rho, tr_P, color='#d1495b', linewidth=1.7, label='Динамическое сжатие')
    ax.scatter([rho], [P_smooth], color='#d1495b', s=55, zorder=5)
    ax.set_title('Неравновесное изотермическое сжатие LJ-флюида', fontsize=10)
    ax.set_xlabel('rho')
    ax.set_ylabel('P')
    x_min = 0.0
    x_max = max(float(RHO_END) * 1.03, float(stationary_isotherm['rho_target'].max()) * 1.03)
    p_values = np.concatenate([stationary_isotherm['P_mean'].to_numpy(), trace_df['P_smooth'].to_numpy()])
    y_pad = max(0.05, 0.08 * (float(np.nanmax(p_values)) - float(np.nanmin(p_values))))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(float(np.nanmin(p_values)) - y_pad, float(np.nanmax(p_values)) + y_pad)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='best')
    info = 'T = ' + format(T_inst, '.3f') + '\n' + 'rho = ' + format(rho, '.3f') + '\n' + 'P_smooth = ' + format(P_smooth, '.3f') + '\n' + segment
    ax.text(0.04, 0.96, info, transform=ax.transAxes, va='top', ha='left', fontsize=9, bbox=dict(facecolor='white', edgecolor='#dddddd', alpha=0.92))
    ax.text(0.04, 0.04, 'Неравновесная траектория; стационарная изотерма дана как ориентир', transform=ax.transAxes, fontsize=7, color='#444444')
    fig.tight_layout(pad=1.2)
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)
    return Image.fromarray(img).resize((RIGHT_WIDTH, VIDEO_HEIGHT))


def compose_frame(frame_index):
    positions = frame_positions[int(frame_index)]
    step = int(frame_steps[int(frame_index)])
    rho = float(frame_rhos[int(frame_index)])
    L = float(frame_Ls[int(frame_index)])
    P_smooth, T_inst, segment = pressure_at_step(step)
    left = render_3d_panel(positions, L)
    right = render_plot_panel(step, rho, P_smooth, T_inst, segment)
    canvas = Image.new('RGB', (VIDEO_WIDTH, VIDEO_HEIGHT), 'white')
    canvas.paste(left.convert('RGB'), (0, 0))
    canvas.paste(right.convert('RGB'), (LEFT_WIDTH, 0))
    draw = ImageDraw.Draw(canvas)
    draw.text((18, 18), 'Неравновесное изотермическое сжатие LJ-флюида', fill=(35, 35, 35))
    draw.text((18, 42), 'step=' + str(step) + '   rho=' + format(rho, '.3f') + '   P_smooth=' + format(P_smooth, '.3f'), fill=(35, 35, 35))
    return canvas


def save_preview_frames():
    if len(frame_steps) < 3:
        indices = [0, max(0, len(frame_steps) // 2), max(0, len(frame_steps) - 1)]
    else:
        target_rhos = [RHO_START, 0.35, RHO_END]
        indices = [int(np.argmin(np.abs(frame_rhos - target))) for target in target_rhos]
    names = ['compression_preview_start.png', 'compression_preview_middle.png', 'compression_preview_end.png']
    for idx, name in zip(indices, names):
        path = OUTPUT_DIR / name
        compose_frame(idx).save(path)
        print('saved:', path)

save_preview_frames()


In [ ]:
# Полный MP4-рендеринг. Можно перезапускать с RUN_SIMULATION=False.


def render_mp4():
    if not RUN_RENDERING:
        print('RUN_RENDERING=False; skipping MP4 render')
        return
    if subprocess.run(['ffmpeg', '-version'], text=True, capture_output=True).returncode != 0:
        raise RuntimeError('ffmpeg is not available')
    render_start = time.perf_counter()
    FRAMES_DIR.mkdir(parents=True, exist_ok=True)
    for old in FRAMES_DIR.glob('frame_*.png'):
        old.unlink()

    frame_paths = []
    hold_frames = int(round(FPS * HOLD_SECONDS))
    sequence = [0] * hold_frames + list(range(len(frame_steps))) + [len(frame_steps) - 1] * hold_frames
    if MAX_RENDER_FRAMES is not None:
        sequence = sequence[:int(MAX_RENDER_FRAMES)]
        if not sequence:
            raise RuntimeError('MAX_RENDER_FRAMES left no frames to render')
        print('Smoke render frames:', len(sequence))
    for out_idx, frame_idx in enumerate(tqdm(sequence, desc='rendering frames')):
        path = FRAMES_DIR / ('frame_' + format(out_idx, '05d') + '.png')
        compose_frame(frame_idx).save(path)
        frame_paths.append(path)

    cmd = [
        'ffmpeg', '-y', '-framerate', str(FPS),
        '-i', str(FRAMES_DIR / 'frame_%05d.png'),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-r', str(FPS),
        str(MP4_PATH),
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError('ffmpeg failed')
    render_time = time.perf_counter() - render_start
    print('saved:', MP4_PATH)
    print('render wall time, s:', format(render_time, '.1f'))
    if not KEEP_RENDERED_FRAMES:
        shutil.rmtree(FRAMES_DIR)
        print('removed temporary frames:', FRAMES_DIR)

render_mp4()


In [ ]:
# Финальная сводка созданных файлов.

print('OUTPUT_DIR:', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob('*')):
    if path.is_file():
        print(path.name, path.stat().st_size, 'bytes')
print('Done. Для повторного рендера без MD установите RUN_SIMULATION = False и перезапустите ячейки начиная с загрузки данных/рендеринга.')
